# Rung 2 follow-up — the protocol-ablation phase diagram (issue #168)

The positive control grokked with a **sparse** circuit (2026-10-14), so the boundary
question is answerable by ablation instead of speculation: starting from the
Nanda-faithful setup, switch on **one deviation at a time** (minibatch, default
betas, cosine LR, embedding renorm, LayerNorm, grad clip) plus the weight-decay axis.

The sweep runner delegates to exp2's own code, budgets in optimizer steps, and records
the post-audit energy metric; every cell emits a citable per-cell manifest
(`--emit-manifests results/`). Cells already completed are skipped on re-run, and the
working `.pt` manifest carries the in-progress state.

**Before running:** GPU T4/P100 + Internet On. Total ~1.5-2h for the core config on a T4.
If the session dies mid-sweep, re-run the cell: completed cells are skipped and the
remaining ones continue.

In [ ]:
BRANCH = "v1.1.0-rc2"  # pin a un tag immutabile: mai main (resume exp1 e _git_sha_exists)
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
# Fail prima di spendere tempo di GPU se il clone è sporco o non punta
# esattamente al tag immutabile. pin_remote_run.py rifiuta git_dirty=True
# e _git_sha_exists richiede che il commit esista localmente.
import subprocess
status = subprocess.run(["git", "status", "--porcelain=v1"], capture_output=True, text=True).stdout
assert not status, f"il clone è sporco:\n{status}"
describe = subprocess.run(["git", "describe", "--tags", "--exact-match"], capture_output=True, text=True).stdout.strip()
assert describe == BRANCH, f"il clone è su {describe!r}, non sul tag immutabile {BRANCH!r}"
print(f"clone pulito e pinned su {describe}")


In [ ]:
# The GPU images ship the CUDA torch stack plus numpy, matplotlib, tqdm, pyyaml;
# `uv sync --frozen` would re-resolve ~5GB of CUDA wheels into the limited working
# disk and die mid-download. Fill only real gaps; the manifests record the versions used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}
import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])

In [ ]:
# The sweep. Resumable: re-run this cell after a disconnect — completed cells are
# skipped (the working manifest records them) and the rest continue.
!python -m scripts.phase_diagram_sweep --config configs/phase_diagram_ablation.yaml --resume --emit-manifests results/


In [ ]:
# Which cells reached the sparse regime, by the post-audit energy criterion?
# Gate first: mean val_acc >= 0.99 before any SPARSE / MIXED / DENSE label.
# SPARSE = every seed sparse (energy_sparse mean == 1.0); MIXED = 1 or 2;
# DENSE = 0. NOT_GROKKED = gate failed, no sparsity label applies.
import json
from pathlib import Path

n_sparse = n_mixed = n_dense = n_not_groked = n_total = 0
for p in sorted(Path('results').glob('phase_diagram_*.json')):
    m = json.load(open(p))
    n_total += 1
    cell = m.get('args', {})
    a = m.get('aggregate', {})
    k90 = a.get('k_energy_90', {}).get('mean')
    sparse = a.get('energy_sparse', {}).get('mean')
    val = a.get('final_val_acc', {}).get('mean')
    cell_id = cell.get('cell_id', 'unknown')
    modulus = cell.get('modulus', '?')
    wd = cell.get('weight_decay', '?')
    if val is None or val < 0.99:
        regime = 'NOT_GROKKED'
        n_not_groked += 1
    elif sparse == 1.0:
        regime = 'SPARSE'
        n_sparse += 1
    elif sparse == 0.0:
        regime = 'DENSE'
        n_dense += 1
    else:
        regime = 'MIXED'
        n_mixed += 1
    print(f'{cell_id:<24} P={modulus:<4} wd={wd:<4} '
          f'val={val if val is not None else "NA":>6} '
          f'k_energy_90={k90 if k90 is not None else "NA":>6} '
          f'sparse_seeds={sparse if sparse is not None else "NA"} '
          f'regime={regime}')
print("\nGate readings (threshold 0.99, primary = cell mean):")
print(f'  SPARSE={n_sparse}/{n_total}  MIXED={n_mixed}/{n_total} '
      f'DENSE={n_dense}/{n_total}  NOT_GROKKED={n_not_groked}/{n_total}')
print(f'  DENSE=0 invariant under every reduction.')


In [ ]:
import glob

from scripts.zip_manifests import write_manifest_zip

manifests = sorted(glob.glob("results/phase_diagram_*.json"))
print("manifests:", len(manifests))
assert manifests, "no manifests produced - nothing to zip"

# See scripts/zip_manifests.py: the old shell-zip line passed a list repr.
archive = write_manifest_zip(manifests, "phase_sweep_results.zip")
print("wrote", archive, "with", len(manifests), "manifest(s)")

try:
    from google.colab import files

    files.download("phase_sweep_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")


## After downloading

1. Pin: `python scripts/pin_remote_run.py --from <dir or zip>` (fail-closed: provenance, seeds/aggregate contract).
2. Cite the per-cell manifests in `portfolio/RESULTS.md` with `<!-- manifest: results/phase_diagram_<cell>.json -->` and run `make verify-claims`.
3. Report the boundary as measured — including every cell that stayed dense.